# 10장. 틀린 만큼 고치기: 손실과 경사 하강

얼마나 틀렸는지 재는 손실, 어느 쪽으로 고칠지 알려 주는 기울기, 그리고 조금씩 고쳐 나가는 경사 하강을 직접 계산해 봅니다.

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

## 실습 10-1 정답에 준 확률로 손실 재기

후보 글자 세 개의 점수가 [2, 1, 0]이고, 정답은 0번 글자입니다.

In [ ]:
import torch
from torch.nn import functional as F

scores = torch.tensor([[2.0, 1.0, 0.0]])    # 문제 1개, 후보 글자 3개의 점수
answer = torch.tensor([0])                  # 정답은 0번 글자

probs = F.softmax(scores, dim=-1)
print("확률:", probs)
print("-log(정답 확률):", -torch.log(probs[0, 0]))
print("cross_entropy:", F.cross_entropy(scores, answer))

## 실습 10-2 확률에 따라 달라지는 손실

**먼저 예상해 보세요.** 정답에 준 확률이 작아지면 손실은 어떻게 될까요?

In [ ]:
for p in [0.9, 0.5, 0.1, 1 / 2453, 0.0001]:
    loss = -torch.log(torch.tensor(p))
    print(f"정답 확률 {p:.4f} → 손실 {loss:.2f}")

## 실습 10-3 문제 여러 개의 손실

문제 세 개의 손실을 하나하나 구해 평균을 내고, cross_entropy와 비교합니다.

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.0],     # 문제 1
                       [0.0, 3.0, 0.0],     # 문제 2
                       [1.0, 1.0, 1.0]])    # 문제 3
answers = torch.tensor([0, 1, 2])           # 문제마다 정답 번호

probs = F.softmax(scores, dim=-1)
picked = torch.stack([probs[0, 0], probs[1, 1], probs[2, 2]])   # 정답 글자의 확률
print("정답 확률:", picked)
print("문제마다 손실:", -torch.log(picked))
print("손실 평균:", (-torch.log(picked)).mean())
print("cross_entropy:", F.cross_entropy(scores, answers))

## 실습 10-4 숫자 하나짜리 모델의 손실

받은 숫자의 3배를 맞히는 문제입니다. 모델은 w × x를 예측합니다. w에 따라 손실이 어떻게 달라질까요?

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])          # 문제
y = 3 * x                                  # 정답: 문제의 3배

def loss_at(w):
    return ((w * x - y) ** 2).mean()

for w in [0.0, 1.0, 2.0, 3.0, 4.0]:
    print(f"w = {w}: 손실 {loss_at(w):.2f}")

## 실습 10-5 기울기 구하기

w를 0에서 아주 조금 늘려 보면 손실이 어느 쪽으로 얼마나 변하는지 알 수 있습니다. PyTorch의 backward가 구한 값과 비교합니다.

In [ ]:
slope = (loss_at(0.01) - loss_at(0.0)) / 0.01
print(f"어림한 기울기: {slope:.2f}")

w = torch.tensor(0.0, requires_grad=True)
loss = ((w * x - y) ** 2).mean()
loss.backward()
print(f"backward로 구한 기울기: {w.grad:.2f}")

## 실습 10-6 경사 하강 다섯 걸음

기울기의 반대쪽으로 조금씩 고쳐 나갑니다.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)  # 고칠 숫자 하나, 0에서 시작
lr = 0.05                                  # 학습률(한 걸음의 크기)

for i in range(5):
    loss = ((w * x - y) ** 2).mean()       # 예측하고 손실 재기
    loss.backward()                        # 기울기 구하기
    print(f"걸음 {i + 1}: w {w.item():.3f} | 손실 {loss.item():.3f} | 기울기 {w.grad.item():.3f}")
    with torch.no_grad():
        w -= lr * w.grad                   # 기울기 반대쪽으로 고치기
    w.grad.zero_()                         # 다음 걸음을 위해 기울기 지우기
print(f"다섯 걸음 뒤: w {w.item():.3f}")

## 실습 10-7 옵티마이저와 학습률

고치기와 지우기를 옵티마이저에게 맡기고, 학습률을 세 가지로 바꿔 봅니다.

**먼저 예상해 보세요.** 학습률을 0.25로 크게 잡으면 3에 더 빨리 다가갈까요?

In [ ]:
def fit(lr, steps=5):
    w = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([w], lr=lr)
    for _ in range(steps):
        loss = ((w * x - y) ** 2).mean()   # ① 예측하고 손실 재기
        optimizer.zero_grad()              # ② 지난 기울기 지우기
        loss.backward()                    # ③ 기울기 구하기
        optimizer.step()                   # ④ 고치기
    return w.item()

for lr in [0.01, 0.05, 0.25]:
    print(f"학습률 {lr}: 다섯 걸음 뒤 w = {fit(lr):.3f}")